# P4-1 · Bài 4 · Làm việc với LLM

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q anthropic

## Lý thuyết

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab (gọi API thật, mất phí theo token)
import anthropic
client = anthropic.Anthropic()                       # đọc ANTHROPIC_API_KEY từ biến môi trường
resp = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1024,
    system="Bạn là trợ lý chăm sóc khách hàng của một shop nội thất. Trả lời ngắn gọn, lịch sự.",
    messages=[{"role": "user", "content": "Bàn trang điểm có kèm gương không?"}],
)
print(resp.content[0].text)
print(resp.stop_reason, resp.usage.input_tokens, resp.usage.output_tokens)

In [ ]:
from pydantic import BaseModel, ValidationError, Field
class Invoice(BaseModel):
    vendor: str
    date: str = Field(description="YYYY-MM-DD")
    total_vnd: int
    items: list[str]

def extract(text, retries=2):
    msg = (f"Trích xuất hóa đơn thành JSON đúng schema sau, chỉ trả JSON:\n{Invoice.model_json_schema()}\n"
           f"<document>{text}</document>")
    history = [{"role": "user", "content": msg}]
    for _ in range(retries + 1):
        out = client.messages.create(model="claude-haiku-5-5", max_tokens=800, messages=history).content[0].text
        try:
            return Invoice.model_validate_json(out.strip().strip("`").removeprefix("json"))   # bỏ khung code nếu có
        except ValidationError as e:
            history += [{"role": "assistant", "content": out},
                        {"role": "user", "content": f"JSON chưa hợp lệ: {e}. Sửa và chỉ trả JSON."}]
    raise ValueError("không trích xuất được")

# phần kiểm tra schema chạy được ngay, không cần gọi API:
ok = Invoice.model_validate_json('{"vendor": "Nội thất An Phát", "date": "2026-10-02", "total_vnd": 1500000, "items": ["bàn trang điểm"]}')
print(ok.total_vnd, ok.items)
try:
    Invoice.model_validate_json('{"vendor": "An Phát", "total_vnd": "một triệu rưỡi"}')
except ValidationError as e:
    print(e.error_count(), "lỗi ở các trường:", [err["loc"][0] for err in e.errors()])   # lỗi này được gửi lại cho model

1500000 ['bàn trang điểm']
3 lỗi ở các trường: ['date', 'total_vnd', 'items']


In [ ]:
# Cần ANTHROPIC_API_KEY và client ở ô đầu: chạy trên máy bạn hoặc Colab
tools = [{
    "name": "get_order_status",
    "description": "Tra trạng thái đơn hàng theo mã đơn. Dùng khi khách hỏi đơn của họ đang ở đâu hoặc khi nào tới.",
    "input_schema": {"type": "object",
                     "properties": {"order_id": {"type": "string", "description": "Mã đơn, dạng DH + số, ví dụ DH1024"}},
                     "required": ["order_id"]},
}]
messages = [{"role": "user", "content": "Đơn DH1024 của tôi tới đâu rồi?"}]
resp = client.messages.create(model="claude-haiku-5-5", max_tokens=500, tools=tools, messages=messages)
while resp.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for call in [b for b in resp.content if b.type == "tool_use"]:      # có thể nhiều lời gọi cùng lúc
        out = TOOLS[call.name](**call.input)                            # TOOLS: dict tên → hàm Python của bạn
        results.append({"type": "tool_result", "tool_use_id": call.id, "content": str(out)})
    messages.append({"role": "user", "content": results})
    resp = client.messages.create(model="claude-haiku-5-5", max_tokens=500, tools=tools, messages=messages)
print(resp.content[0].text)

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab
with client.messages.stream(model="claude-haiku-5-5", max_tokens=1024, messages=messages) as s:
    for text in s.text_stream:
        print(text, end="", flush=True)

In [ ]:
import random, time
def call_with_retry(fn, max_tries=6):
    for k in range(max_tries):
        try:
            return fn()
        except anthropic.APIStatusError as e:
            retryable = e.status_code == 429 or e.status_code >= 500     # rate limit hoặc lỗi phía máy chủ
            if not retryable or k == max_tries - 1:
                raise                                                     # lỗi 400 (yêu cầu sai) thì không thử lại
        except anthropic.APIConnectionError:                              # mất mạng, timeout
            if k == max_tries - 1:
                raise
        time.sleep(min(30, 2 ** k) * random.uniform(0.5, 1.0))

# thử với một hàm giả lập: hai lần đầu bị rate limit (429), lần ba thành công
import anthropic, httpx
calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise anthropic.RateLimitError("rate limited", response=httpx.Response(429, request=httpx.Request("POST", "https://api.anthropic.com")), body=None)
    return "ok"
t0 = time.time()
print(call_with_retry(flaky), "sau", calls["n"], "lần gọi,", round(time.time() - t0, 1), "giây chờ")

ok sau 3 lần gọi, 2.3 giây chờ


## Bài tập 5: Tỉ lệ thành công của trích xuất {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần ANTHROPIC_API_KEY và file invoices_eval.json của bạn
import json, time

def evaluate(samples):
    ok_json = field_ok = 0
    fields = ["vendor", "date", "total_vnd"]
    t0 = time.time()
    for s in samples:
        try:
            inv = extract(s["text"])
            ok_json += 1
            field_ok += sum(getattr(inv, f) == s["gold"][f] for f in fields)
        except ValueError:
            pass
    n = len(samples)
    print(f"JSON hợp lệ {ok_json / n:.0%} | đúng từng trường {field_ok / (n * len(fields)):.0%} | {(time.time() - t0) / n:.1f}s mỗi mẫu")

samples = json.load(open("invoices_eval.json", encoding="utf-8"))   # [{"text": ..., "gold": {...}}, ...]
evaluate(samples)